# 03 · Validate and split the contrast pairs

Validates the reviewed pairs (docs/04 §2), checks they do not leak into evaluation sets, and splits them into calibration / probe / validation. With no reviewed file attached it builds an **unpaired fallback** (harmful pool + Alpaca prompts) so 04–06 can start; rerun later with real pairs.

| | |
|---|---|
| **Kaggle settings** | Accelerator: None (CPU is enough) · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · output of **01** · `safetypulse-pairs` (when ready) |
| **Reads** | `pairs_reviewed.csv`, `data/*.jsonl` |
| **Writes** (`/kaggle/working/`) | `pairs/pairs.jsonl`, `pairs/pairs_manifest.json` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [1]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

{'torch': '2.11.0+cpu',
 'transformers': '5.20.2',
 'cuda_available': False,
 'on_kaggle': True,
 'smoke': False,
 'peft': '0.21.2'}

In [2]:
import random
from collections import Counter

SEED = 0
N_PROBE, N_VAL = (50, 50) if not sp.SMOKE else (4, 4)
LEN_RATIO = (0.8, 1.25)
OUT = sp.out_dir("pairs")
eval_harm = sp.load_jsonl(sp.find_input("eval_harm.jsonl"))
eval_or = sp.load_jsonl(sp.find_input("eval_overrefusal.jsonl"))
reviewed = sp.find_input("pairs_reviewed.csv", required=False)
print("reviewed pairs:", reviewed)

reviewed pairs: /kaggle/input/datasets/purvanshjoshi1/safetypulse-pairs/pairs_reviewed.csv


In [3]:
log = Counter()
if reviewed:
    matched = True
    rows = sp.load_csv(reviewed)
    pairs = []
    for r in rows:
        if str(r.get("keep", "")).strip().lower() in ("n", "no", "0", "false"):
            log["dropped: keep=n"] += 1
            continue
        h, s = r["harmful"].strip(), r.get("harmless", "").strip()
        if not s or s.lower() == h.lower():
            log["dropped: empty or unchanged twin"] += 1
            continue
        ratio = len(s.split()) / max(1, len(h.split()))
        if not LEN_RATIO[0] <= ratio <= LEN_RATIO[1]:
            log["dropped: length ratio"] += 1
            continue
        pairs.append({"pair_id": r["pair_id"], "category": r.get("category") or "unknown", "harmful": h,
                      "harmless": s, "source": r.get("source", "advbench") + "|" + (r.get("reviewer") or "team"),
                      "annotator_2_valid": r.get("annotator_2_valid", "")})
else:
    matched = False
    print("!! No pairs_reviewed.csv: building the UNPAIRED fallback. Matched-pair results (C2) need real pairs.")
    pool = sp.load_jsonl(sp.find_input("pairs_pool.jsonl"))
    harmless = sp.load_jsonl(sp.find_input("harmless_pool.jsonl"))
    pairs = [{"pair_id": p["pair_id"], "category": "unknown", "harmful": p["harmful"], "harmless": s["prompt"],
              "source": "advbench|alpaca-unpaired"} for p, s in zip(pool, harmless)]
print(len(pairs), "pairs", dict(log))

288 pairs {'dropped: keep=n': 12}


In [4]:
# Second-annotator agreement (Cohen's kappa) if that column was filled
a2 = [p for p in pairs if str(p.get("annotator_2_valid", "")).strip()]
if a2:
    # annotator 1 kept every remaining pair (valid = y); kappa against annotator 2 over all reviewed rows
    rows_all = sp.load_csv(reviewed)
    y1 = [str(r.get("keep", "y")).strip().lower() not in ("n", "no", "0", "false") for r in rows_all if r.get("annotator_2_valid")]
    y2 = [str(r["annotator_2_valid"]).strip().lower() in ("y", "yes", "1", "true") for r in rows_all if r.get("annotator_2_valid")]
    po = sum(a == b for a, b in zip(y1, y2)) / len(y1)
    pe = (sum(y1) / len(y1)) * (sum(y2) / len(y2)) + (1 - sum(y1) / len(y1)) * (1 - sum(y2) / len(y2))
    kappa = (po - pe) / (1 - pe) if pe < 1 else 1.0
    print(f"Cohen's kappa = {kappa:.2f} over {len(y1)} pairs")
    disagree = {r["pair_id"] for r, a, b in zip([r for r in rows_all if r.get("annotator_2_valid")], y1, y2) if a != b}
    pairs = [p for p in pairs if p["pair_id"] not in disagree]
    print(f"dropped {len(disagree)} pairs with disagreement")
else:
    kappa = None
    print("No second annotator: report this as a limitation (docs/04 §2).")
print("categories:", Counter(p["category"] for p in pairs).most_common())

No second annotator: report this as a limitation (docs/04 §2).
categories: [('cyber', 92), ('fraud', 64), ('other', 62), ('weapons', 25), ('hate', 17), ('chemicals', 15), ('self-harm', 13)]


In [5]:
leak_h = sp.near_duplicate_pairs([p["harmful"] for p in pairs], [x["prompt"] for x in eval_harm], 0.5)
leak_s = sp.near_duplicate_pairs([p["harmless"] for p in pairs], [x["prompt"] for x in eval_or], 0.8)
bad = {i for i, _, _ in leak_h} | {i for i, _, _ in leak_s}
pairs = [p for i, p in enumerate(pairs) if i not in bad]
print(f"removed {len(bad)} pairs that overlap evaluation sets")

rng = random.Random(SEED)
rng.shuffle(pairs)
for i, p in enumerate(pairs):
    p["split"] = "probe" if i < N_PROBE else "val" if i < N_PROBE + N_VAL else "calib"
    p["matched"] = matched
assert sum(p["split"] == "calib" for p in pairs) >= max(4, N_PROBE), "too few pairs left for calibration"
path = sp.save_jsonl(pairs, OUT / "pairs.jsonl")
sp.save_json({"matched": matched, "n": len(pairs), "splits": Counter(p["split"] for p in pairs),
              "categories": Counter(p["category"] for p in pairs), "kappa": kappa, "dropped": dict(log),
              "sha256": sp.sha256_file(path), "provenance": sp.provenance(SEED)}, OUT / "pairs_manifest.json")
Counter(p["split"] for p in pairs)

removed 0 pairs that overlap evaluation sets


Counter({'probe': 50, 'val': 50, 'calib': 188})